In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# 设置绘图风格与中文字体
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWD 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_price_year_data(city_name):
    """
    获取指定城市的总价与建成年份数据。
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        total_price,
        build_year
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city = '{city_name}' 
      AND total_price IS NOT NULL 
      AND build_year IS NOT NULL
      AND build_year > 1900 
      AND build_year <= 2026
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_build_year_barh(city):
    """绘制色彩渐变水平条形图"""
    df = fetch_price_year_data(city)
    
    if df.empty or len(df) < 20:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 有效建成年份数据不足，无法生成分析图", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 动态价格分箱 ---
    quantiles = df['total_price'].quantile([0.2, 0.4, 0.6, 0.8]).tolist()
    bins = [-1] + [q + (i * 0.0001) for i, q in enumerate(quantiles)] + [float('inf')]
    
    labels = [
        f'低价位 (<{int(quantiles[0])}万)', 
        f'中低价 ({int(quantiles[0])}-{int(quantiles[1])}万)', 
        f'中等价 ({int(quantiles[1])}-{int(quantiles[2])}万)', 
        f'中高价 ({int(quantiles[2])}-{int(quantiles[3])}万)', 
        f'高价位 (>{int(quantiles[3])}万)'
    ]
    df['price_tier'] = pd.cut(df['total_price'], bins=bins, labels=labels)

    # --- 2. 计算各价格区间的平均建成年份 ---
    stats = df.groupby('price_tier')['build_year'].mean().reset_index()
    # 删除没有数据的区间
    stats = stats.dropna()

    if stats.empty:
        return

    # --- 3. 颜色渐变映射逻辑 ---
    # 使用 Normalize 将年份数值映射到 0-1 之间
    # 年份越新（数值越大），颜色越深/越暖
    min_year = stats['build_year'].min()
    max_year = stats['build_year'].max()
    
    # 防止所有区间年份完全一样导致 Normalize 报错
    if max_year - min_year < 1:
        min_year -= 1
        max_year += 1
        
    norm = mcolors.Normalize(vmin=min_year - 2, vmax=max_year + 2)
    cmap = cm.get_cmap('YlGnBu') 
    colors = cmap(norm(stats['build_year']))

    # --- 4. 开始绘图 ---
    fig, ax = plt.subplots(figsize=(12, 6))
    
    y_pos = np.arange(len(stats['price_tier']))
    means = stats['build_year']

    # 绘制水平条形图 (barh)
    bars = ax.barh(y_pos, means, color=colors, edgecolor='white', height=0.6)

    # --- 5. 标注与修饰 ---
    ax.set_xlim(min_year - 3, max_year + 3)

    # 在条形图的末端加上具体的年份文字标注
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(
            width + 0.3, 
            bar.get_y() + bar.get_height()/2, 
            f'{int(width)}年', 
            ha='left', va='center', fontsize=12, fontweight='bold', color='#333333'
        )

    plt.title(f'[{city}] 各价格区间房源平均建成年份', fontsize=16, pad=20)
    plt.xlabel('平均建成年份 (数值放大展示)', fontsize=12, labelpad=10)
    plt.ylabel('自适应价格区间', fontsize=12)
    
    # 设置 Y 轴刻度标签
    ax.set_yticks(y_pos)
    ax.set_yticklabels(stats['price_tier'], fontsize=11)
    
    # 添加解释性文字
    ax.text(
        0.98, 0.05, 
        '视觉引导：颜色越深，代表该区间房源整体越新', 
        transform=ax.transAxes, ha='right', va='bottom', 
        fontsize=10, color='gray', style='italic'
    )

    # 辅助网格线与去边框
    ax.grid(axis='x', linestyle=':', alpha=0.6)
    sns.despine(left=True, bottom=True)

    plt.tight_layout()
    plt.show()

In [3]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_build_year_barh(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_build_year_barh(default_city)

In [4]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()